In [ ]:
# Google Colab Setup
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab. Installing dependencies...")
    !pip install -q warp-lang pydantic pyyaml scipy matplotlib
    if not os.path.exists('pazuzu'):
        !git clone https://github.com/dzilles/pazuzu.git
    
    repo_path = os.path.abspath('pazuzu')
    if repo_path not in sys.path:
        sys.path.append(repo_path)
    print(f"Project root added to sys.path: {repo_path}")
else:
    print("Running locally. Skipping Colab setup.")

# Isentropic Vortex Verification

This notebook validates the High-Order Flux Reconstruction solver using the Isentropic Vortex test case.

## Euler Equations
We solve the 2D Compressible Euler equations:
$$\frac{\partial \mathbf{q}}{\partial t} + \nabla \cdot \mathbf{F}(\mathbf{q}) = 0$$
where $\mathbf{q} = [\rho, \rho u, \rho v, E]^T$.

## Exact Solution
The isentropic vortex is an exact solution of the Euler equations representing a perturbation convecting in a freestream.
The perturbations are given by:
$$ \delta u = -\frac{S}{2\pi} (y-y_c) e^{(1-r^2)/2} $$
$$ \delta v = \frac{S}{2\pi} (x-x_c) e^{(1-r^2)/2} $$
$$ T = 1 - \frac{(\gamma-1)S^2}{8\pi^2} e^{1-r^2} $$


In [ ]:
import sys
import os
import numpy as np
try:
    import warp as wp
except ImportError:
    print("Error: warp-lang not found. Please install it with 'pip install warp-lang' or use the .venv environment.")
import matplotlib.pyplot as plt

# Add project root to path (robust discovery)
def find_project_root():
    # Check if we already set it in Colab cell
    if 'repo_path' in globals():
        return repo_path
        
    curr = os.getcwd()
    # Search upwards for solver.py to find the root
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'solver.py')):
            return curr
        curr = os.path.dirname(curr)
    # Fallback to relative path
    return os.path.abspath("../../../")

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added to sys.path: {project_root}")

try:
    from solver import PazuzuSolver
    from src.kernels.initial_conditions import init_isentropic_vortex
    print("Successfully imported Solver modules.")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Ensure you are running this notebook from within the project structure.")

In [ ]:
config_path = os.path.join(project_root, "tests/verification/vortex_2D/vortex.yaml")
if not os.path.exists(config_path):
    # Fallback for local run if already in the directory
    config_path = "vortex.yaml"

solver = PazuzuSolver(config_path)
print(f'Initialized solver with N={solver.basis.N} (Order {solver.basis.N+1})')

In [ ]:
# Visualize Initial Density and Velocity
q = solver.state.q.numpy()
rho = q[:, :, 0]
u = q[:, :, 1] / rho
v = q[:, :, 2] / rho
vel_mag = np.sqrt(u**2 + v**2)

x = solver.state.x.numpy()
y = solver.state.y.numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.scatter(x.flatten(), y.flatten(), c=rho.flatten(), cmap='viridis', s=2)
plt.colorbar(im1, ax=ax1, label='Density')
ax1.set_title('Initial Density')
ax1.set_aspect('equal')

im2 = ax2.scatter(x.flatten(), y.flatten(), c=vel_mag.flatten(), cmap='magma', s=2)
plt.colorbar(im2, ax=ax2, label='Velocity Magnitude')
ax2.set_title('Initial Velocity Magnitude')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
dt = 0.001
t_final = solver.config.simulation.t_final
t = 0.0
step = 0

print(f"Running to t={t_final}...")
while t < t_final:
    solver.integrator.step_ssp_rk3(
        solver.compute_rhs,
        dt,
        t,
        solver.quadtree.num_blocks
    )
    t += dt
    step += 1
    if step % 10 == 0:
        print(f"Step {step}, t={t:.3f}", end='\r')
print("\nDone.")

In [ ]:
# Compute Exact Solution
q_exact_wp = wp.zeros_like(solver.state.q)
wp.launch(
    kernel=init_isentropic_vortex,
    dim=(solver.quadtree.num_blocks, solver.basis.Np),
    inputs=[
        solver.state.x,
        solver.state.y,
        q_exact_wp,
        solver.state.active_block_indices,
        solver.quadtree.num_blocks,
        solver.params,
        t # Final time
    ],
    device=solver.device
)
q_exact = q_exact_wp.numpy()
q_num = solver.state.q.numpy()

# Compute L2 Error
# Weights
w = solver.basis.weights_1d.numpy()
N1 = solver.basis.N1
w_2d = np.zeros(solver.basis.Np)
for j in range(N1):
    for i in range(N1):
        w_2d[j*N1 + i] = w[i] * w[j]

# Jacobian (Uniform Grid approximation)
level = solver.config.amr.initial_depth
grid_dim = 1 << level
h = 2.0 / grid_dim
detJ = (h / 2.0)**2

diff_rho = q_num[:, :, 0] - q_exact[:, :, 0]
error_sq = 0.0
for b in range(solver.quadtree.num_blocks):
    for n in range(solver.basis.Np):
        error_sq += diff_rho[b, n]**2 * w_2d[n] * detJ

L2_error = np.sqrt(error_sq)
print(f'L2 Error: {L2_error:.6e}')


## Results Analysis
The L2 error should be small (typically < 1e-2 for coarse grids, < 1e-4 for fine grids).
For N=3 (4th order), the error should decay as $O(h^4)$.